# Is this Elo model actually calibrated?

A probability is only worth something if it means what it says. This notebook takes the free 2,000-row sample and checks, from scratch, whether the model's stated confidence matches what actually happened.

Nothing is taken on trust — every number is recomputed from the CSV. **Spoiler: it is not perfectly calibrated, and the notebook shows exactly how it fails.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

m = pd.read_csv("../data/tt_ratings_sample_2000.csv", parse_dates=["date"])
print(f"{len(m):,} matches | {m.date.min().date()} to {m.date.max().date()} | {m.league.nunique()} competitions")
m.head()

## 1. The columns that matter

`elo_a_pre` and `elo_b_pre` are the two players' ratings **before** the match. `p_model` is the model's probability that player A wins, derived from those ratings alone. `winner` is what actually happened.

The entire value of the dataset rests on one property: nothing in a `_pre` column may depend on the result.

In [ ]:
m[["date", "league", "elo_a_pre", "elo_b_pre", "p_model", "winner", "sets_a", "sets_b"]].head(8)

## 2. Sanity check — is there a side bias?

If player A won far more or less than half the time, the "A" slot would carry information by itself and every probability would be suspect.

In [ ]:
y = (m.winner == "A").astype(int)
print(f"player A wins {y.mean():.1%} of matches")

## 3. Scoring

The Brier score is the mean squared error of a probabilistic forecast — lower is better. Always predicting 50/50 scores 0.2500, so anything at or above that means the model adds nothing.

In [ ]:
brier_model = ((m.p_model - y) ** 2).mean()
brier_naive = ((0.5       - y) ** 2).mean()

print(f"model         {brier_model:.4f}")
print(f"always 50/50  {brier_naive:.4f}")
print(f"improvement   {brier_naive - brier_model:+.4f}")

## 4. The real test: calibration

Brier rewards being confident *and* right, so a poorly calibrated model can still score acceptably. Calibration asks the sharper question: **when the model says 70%, does it win 70% of the time?**

We bucket predictions by stated confidence and compare against reality. We also compute a **Wilson 95% confidence interval** for each bucket — with only a few dozen matches in the high bands, the observed rate carries real uncertainty, and pretending otherwise would be the exact sin this dataset is meant to avoid.

In [ ]:
def wilson(k, n, z=1.96):
    """Wilson score interval — behaves properly for small n, unlike the normal approximation."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2*n)) / d
    h = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / d
    return (max(0, c - h) * 100, min(1, c + h) * 100)

d = m.copy()
d["conf"] = np.maximum(d.p_model, 1 - d.p_model)     # confidence in whichever player it favours
d["hit"]  = (d.p_model > 0.5) == (d.winner == "A")    # was the favourite right?

bands = [(.50,.55), (.55,.60), (.60,.65), (.65,.70), (.70,.75), (.75,.80), (.80,1.01)]

rows = []
for lo, hi in bands:
    s = d[(d.conf >= lo) & (d.conf < hi)]
    if len(s) == 0:
        continue
    k, n = int(s.hit.sum()), len(s)
    lo_ci, hi_ci = wilson(k, n)
    rows.append({
        "band":     f"{lo*100:.0f}-{min(hi,1)*100:.0f}%",
        "n":        n,
        "said":     round((lo + min(hi, 1.0)) / 2 * 100, 1),
        "observed": round(k / n * 100, 1),
        "ci_low":   round(lo_ci, 1),
        "ci_high":  round(hi_ci, 1),
    })

cal = pd.DataFrame(rows)
cal["consistent"] = (cal.said >= cal.ci_low) & (cal.said <= cal.ci_high)
cal

The `consistent` column asks: *is the model's stated confidence inside the range this sample can actually distinguish?* A band can look far off and still be consistent with the truth when only 50 matches sit in it — and a band can look fine while hiding a real bias that only a larger sample reveals.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.8), dpi=120)

ax.plot([48, 92], [48, 92], "--", color="gray", lw=1.4, alpha=.7,
        label="perfect calibration", zorder=1)

ax.errorbar(cal.said, cal.observed,
            yerr=[cal.observed - cal.ci_low, cal.ci_high - cal.observed],
            fmt="o", color="#d2791e", ms=8, lw=2, capsize=5, capthick=1.6,
            label="observed (95% CI)", zorder=3)
ax.plot(cal.said, cal.observed, "-", color="#d2791e", lw=1.6, alpha=.55, zorder=2)

for _, r in cal.iterrows():
    ax.annotate(f"n={r.n}", (r.said, r.ci_low), textcoords="offset points",
                xytext=(0, -16), ha="center", fontsize=8, color="gray")

ax.set_xlabel("What the model said (%)")
ax.set_ylabel("What actually happened (%)")
ax.set_title("Stated confidence vs. observed win rate — 2,000-row sample",
             fontweight="bold", loc="left")
ax.legend(frameon=False, loc="upper left")
ax.grid(axis="y", ls=":", alpha=.5)
for s_ in ("top", "right"):
    ax.spines[s_].set_visible(False)
plt.tight_layout()
plt.show()

n_ok = int(cal.consistent.sum())
print(f"{n_ok} of {len(cal)} bands are statistically consistent with perfect calibration on this sample.")

## 5. What the full file says

2,000 rows cannot resolve the high-confidence bands — look at how wide the intervals get above 70%. Measured on the **full 103,561-match file** (102,787 rows once cold-start matches are dropped), the picture is unambiguous:

| Model said | Matches | Actually won | Gap |
|---|---|---|---|
| 50–55% | 43,521 | 51.7% | −0.8 |
| 55–60% | 28,944 | 55.0% | −2.5 |
| 60–65% | 15,848 | 58.9% | −3.6 |
| 65–70% | 7,706 | 63.3% | −4.2 |
| 70–75% | 3,642 | 69.3% | −3.2 |
| 75–80% | 1,801 | 75.0% | −2.5 |
| 80%+ | 1,325 | 79.1% | — |

**Every gap is negative.** The model is systematically over-confident above 60%, by up to 4.2 points. An earlier release measured 2.9 points on half the data; on twice the data it is 4.2.

The bias is concentrated rather than uniform. On matches stated at 65% or above:

| League | Matches | Observed | Stated | Gap |
|---|---|---|---|---|
| TT Elite Series | 8,143 | 70.6% | 72.6% | −2.0 |
| Setka Cup | 4,166 | 63.8% | 70.0% | −6.2 |
| TT Cup | 1,293 | 63.0% | 68.6% | −5.7 |
| Czech Liga Pro | 503 | 55.9% | 67.3% | −11.4 |

Which league you work on matters more than the aggregate does.

## 6. The cold-start caveat

Every player enters at 1500. Until a rating has moved, the model has nothing to say and outputs exactly 0.500. Those rows are not wrong, just uninformative. This sample already excludes them; in the full dataset they are 774 matches, 0.7%, and you filter them like this:

In [ ]:
warm = m[(m.elo_a_pre != 1500) & (m.elo_b_pre != 1500)]
print(f"{len(m) - len(warm)} cold-start rows in this sample ({len(warm)} remain)")

## 7. Where to go next

- Treat `p_model` as a baseline and try to beat it with your own features — given the over-confidence above, that is not a high bar.
- Or ignore `p_model` entirely and use `elo_a_pre`, `elo_b_pre`, `sol_a_pre`, `sol_b_pre` as leak-free inputs to your own classifier. **That is where the durable value of this dataset sits**: pre-match ratings with no look-ahead leakage are the part that is genuinely hard to build.
- Recalibrate per league rather than globally. The table above shows why a single global mapping cannot serve TT Elite Series and Czech Liga Pro at once.

One caution worth repeating: on a clean 13,892-match holdout the set-level model scores **0.2428** against **0.2480** for a match-level Elo at K=32 — an edge of **+0.0052 Brier**. But K=32 is not that baseline's optimum (the methodology puts it near K=14), so the edge over a *fully tuned* match-level Elo is smaller than that, and is not quantified here. This dataset is a well-built input to a model, not an edge in a box.